In [72]:
import random

class NakliLLM:
    def __init__(self):
        print('LLM created')

    def predict(self, prompt):

        response_list = [
            "Delhi is the capital of India",
            "India is the 3rd largest economy in the world",
            "Chatrapati Shivaji Maharaj is the father of Indian Navy",
            "Maharana Pratap defeated Akbar in Haldi Ghati Yudh"
        ]

        return {'response': random.choice(response_list)}

In [73]:
llm = NakliLLM()

LLM created


In [74]:
llm.predict("What is the capital of India?")

{'response': 'Delhi is the capital of India'}

In [75]:
class NakliPromptTemplate:
    def __init__(self, template, input_variables):
        self.template = template
        self.input_variables = input_variables

    def format(self, input_dict):

        return self.template.format(**input_dict)

In [76]:
template = NakliPromptTemplate(
    template = 'Explain about {topic} in {length}',
    input_variables = ['topic', 'length']
)

prompt = template.format({'topic': 'India', 'length': 'short'})

prompt

'Explain about India in short'

In [77]:
# Now we will create an LLM app using these two classes.
# User will give the topic and length and LLM will give its output

In [78]:
llm = NakliLLM()

LLM created


In [79]:
response = llm.predict(prompt)

response

{'response': 'Maharana Pratap efeated Akbar in Haldi Ghati Yudh'}

# Now, let's create an LLM chain 

In [80]:
class NakliLLMChain:
    def __init__(self, llm, prompt):
        self.llm = llm
        self.prompt = prompt

    def invoke(self, input_dict):

        final_prompt = self.prompt.format(input_dict)
        response = self.llm.predict(final_prompt)

        return response
    

In [81]:
prompt = NakliPromptTemplate(
    template = 'Explain about {topic} in {length}',
    input_variables = ['topic', 'length']
)

In [82]:
llm = NakliLLM()

LLM created


In [83]:
chain = NakliLLMChain(llm = llm, prompt = prompt)

print(chain.llm)
print(chain.prompt)

In [89]:
response = chain.invoke({'topic': 'Indian cricket team', 'length': 'short'})

response['response']

'Maharana Pratap efeated Akbar in Haldi Ghati Yudh'

In [1]:
# Now, suppose that we want gtp to generate a joke and gemini to explain that joke?
# Try it if u want, but u will need large number of code's to do this.
# And this is bcz different methods like format() in PromptTemplate and predict() in LLM.

# But, if we keep one common method in all these classes, we cam slve the problem.

# Creating Runnable

In [5]:
# Creating an abstract class

from abc import ABC, abstractmethod

class Runnable(ABC):
    
    @abstractmethod
    def invoke(input):
        pass

In [13]:
import random

class NakliLLM(Runnable):
    def __init__(self):
        print('LLM created')

    def invoke(self, prompt):    # Predict changed to invoke

        response_list = [
            "Delhi is the capital of India",
            "India is the 3rd largest economy in the world",
            "Chatrapati Shivaji Maharaj is the father of Indian Navy",
            "Maharana Pratap defeated Akbar in Haldi Ghati Yudh"
        ]

        return {'response': random.choice(response_list)}

    
    def predict(self, prompt):

        response_list = [
            "Delhi is the capital of India",
            "India is the 3rd largest economy in the world",
            "Chatrapati Shivaji Maharaj is the father of Indian Navy",
            "Maharana Pratap defeated Akbar in Haldi Ghati Yudh"
        ]

        print("This method will be removed in later version, instead use invoke().")
        return {'response': random.choice(response_list)}

In [11]:
llm = NakliLLM()

LLM created


In [14]:
class NakliPromptTemplate(Runnable):
    def __init__(self, template, input_variables):
        self.template = template
        self.input_variables = input_variables

    def invoke(self, input_dict):    # format() to invoke()

        return self.template.format(**input_dict)


    def format(self, input_dict):
        print("This method will be removed in later version, instead use invoke().")
        return self.template.format(**input_dict)

In [131]:
# Runnable connector

class RunnableConnector(Runnable):

    def __init__(self, runnable_list):
        self.runnable_list = runnable_list

    def invoke(self, input_data):

        for runnable in self.runnable_list:

            input_data = runnable.invoke(input_data)
        return input_data

In [132]:
llm = NakliLLM()

LLM created


In [133]:
template = NakliPromptTemplate(
    template = "Explain about {topic} in {length}",
    input_variables = ['topic', 'length']
)

prompt = template.invoke({'topic': 'Indian cricket team', 'length': 'short'})

In [134]:
llm.invoke(prompt)

{'response': 'Delhi is the capital of India'}

In [135]:
# Using chain

In [136]:
chain = RunnableConnector(runnable_list = [template, llm])

chain.invoke({'topic': 'Indian cricket team', 'length': 'short'})

{'response': 'Chatrapati Shivaji Maharaj is the father of Indian Navy'}

In [137]:
# Now, we can add as many runnable we want in this same class.
# Adding StrOutputParser in this runnableconnector class.


class NakliStrOutputParser(Runnable):
    def __init__(self):
        pass
        
    def invoke(self, input_data):

        return input_data['response']

In [138]:
parser = NakliStrOutputParser()

In [139]:
chain = RunnableConnector(runnable_list = [template, llm, parser])

chain.invoke({'topic': 'Indian cricket team', 'length': 'short'})

'India is the 3rd largest economy in the world'

# Now, lets connect two chains together and create one long chain.

`chain_1 will generate a joke and chain_2 will give its explanation`

In [156]:
temp_joke = NakliPromptTemplate(
    template = "Write an joke on {topic}",
    input_variables = ['topic']
)

temp_exp = NakliPromptTemplate(
    template = '''Explain the given joke.
    {response}''',
    input_variables = ['response']
)

In [157]:
llm = NakliLLM()

LLM created


In [158]:
parser = NakliStrOutputParser()

In [159]:
chain_1 = RunnableConnector([temp_joke, llm])

chain_2 = RunnableConnector([temp_exp, llm, parser])

In [160]:
chain_combined = RunnableConnector([chain_1, chain_2])

chain_combined.invoke({'topic': 'Doraemon'})

'India is the 3rd largest economy in the world'